# Instructivo para cliente: Activación de VCStack en Allied Telesis x950

**Equipo:** Allied Telesis AT-x950-28XTQm  
**Función:** Virtual Chassis Stacking / VCStack  
**Cable de stack identificado en BOM:** AT-QSFP28-1CU  
**Tipo de cable:** 100G DAC pasivo, 1 metro  
**Cantidad para stack:** 2 unidades  

Este documento resume qué cable fue comprado para stacking, cómo debe cablearse y qué pasos de configuración se deben ejecutar para activar VCStack en los dos switches x950.


## 1. Cable identificado en el BOM

En el BOM aparecen dos tipos de cables DAC QSFP:

| PN | Descripción | Cantidad | Uso sugerido |
|---|---:|---:|---|
| **AT-QSFP28-1CU** | **100G DAC pasivo, 1 m** | **2** | **Stack VCStack recomendado para x950** |
| AT-QSFP1CU | QSFP+ 40G DAC, 1 m | 4 | Cable 40G, no usar mezclado con el trunk 100G |

Para el stack de los **2 switches AT-x950-28XTQm**, el cable a usar es:

> **AT-QSFP28-1CU — 100G DAC pasivo, 1 metro**

Nota: el manual de VCStack para x950 permite usar cables **QSFP28-1CU / QSFP28-3CU** como enlaces de stack a **100G**, con un máximo de dos puertos 100G por switch para el trunk.


## 2. Topología física propuesta

Usar dos cables **AT-QSFP28-1CU**.

```text
Switch 1 / Master                         Switch 2 / Member
AT-x950-28XTQm                            AT-x950-28XTQm

Puerto 33  =============================  Puerto 33
Puerto 37  =============================  Puerto 37
```

### Cableado recomendado

```text
Switch 1, puerto 33  <---- AT-QSFP28-1CU ---->  Switch 2, puerto 33
Switch 1, puerto 37  <---- AT-QSFP28-1CU ---->  Switch 2, puerto 37
```

Los puertos 33 y 37 son puertos QSFP+/QSFP28 del x950-28XTQm y pueden operar como enlaces 100G para VCStack.


## 3. Consideraciones importantes antes de configurar

Antes de activar el stack:

1. Verificar que ambos switches sean del modelo previsto o compatibles dentro de la serie x950.
2. Verificar que ambos tengan la misma versión del firmware del sistema operativo AlliedWare Plus.
3. No mezclar cables 40G y 100G en el mismo stack trunk.
4. No conectar todavía los cables de stack, o desconectarlos si ya estuvieran conectados.
5. Realizar la configuración inicialmente por consola local.
6. Guardar configuración y reiniciar cuando el procedimiento lo indique.

### Identificación propuesta

| Equipo | Rol | Stack ID | Priority |
|---|---|---:|---:|
| Switch 1 | Master | 1 | 1 |
| Switch 2 | Member | 2 | 2 |

El valor de prioridad más bajo tiene mayor prioridad para ser Master. Por eso se propone **priority 1** para el Switch 1.


## 4. Configuración del Switch 1 — Master

Conectarse por consola al switch que será el Master.

```text
awplus> enable
awplus# show system environment
awplus# show version
```

Verificar estado de hardware y versión de firmware.

Entrar a configuración:

```text
awplus# configure terminal
```

Activar VCStack:

```text
awplus(config)# stack enable
```

Definir prioridad del Master:

```text
awplus(config)# stack priority 1
```

Provisionar el segundo switch:

```text
awplus(config)# switch 2 provision x950-28
```

Guardar y reiniciar:

```text
awplus(config)# exit
awplus# write
awplus# reboot
```


## 5. Configuración del Switch 2 — Member

Conectarse por consola al segundo switch.

```text
awplus> enable
awplus# show system environment
awplus# show version
```

Confirmar que la versión de firmware del AlliedWare Plus coincida con la del Switch 1.

Entrar a configuración:

```text
awplus# configure terminal
```

Activar VCStack:

```text
awplus(config)# stack enable
```

Renumerar el switch como miembro ID 2:

```text
awplus(config)# stack 1 renumber 2
```

Definir prioridad del miembro:

```text
awplus(config)# stack priority 2
```

Guardar y reiniciar:

```text
awplus(config)# exit
awplus# write
awplus# reboot
```

Luego del reinicio, el LED de ID del equipo debería indicar **2**.


## 6. Configuración de puertos 100G y stackport

Después de que ambos switches hayan reiniciado, volver a entrar al **Switch 1 / Master**.

```text
awplus> enable
awplus# configure terminal
```

Configurar los puertos 33 y 37 del Switch 1 como 100G:

```text
awplus(config)# platform portmode interface port1.0.33,port1.0.37 100g
```

Configurar los puertos 33 y 37 del Switch 2 provisionado como 100G:

```text
awplus(config)# platform portmode interface port2.0.33,port2.0.37 100g
```

Asignar los puertos como stack ports:

```text
awplus(config)# interface port1.0.33,port1.0.37
awplus(config-if)# stackport
awplus(config-if)# exit

awplus(config)# interface port2.0.33,port2.0.37
awplus(config-if)# stackport
awplus(config-if)# exit
```

Guardar y reiniciar:

```text
awplus(config)# exit
awplus# write
awplus# reboot
```


## 7. Conexión física de los cables

Una vez configurados los puertos como stack ports, conectar los cables:

```text
Switch 1 port 33  <-->  Switch 2 port 33
Switch 1 port 37  <-->  Switch 2 port 37
```

Luego encender o reiniciar ambos switches y esperar a que el stack se termine de formar.


## 8. Verificación final

Desde el Master:

```text
awplus> enable
awplus# show stack
```

Resultado esperado:

```text
ID   Priority   Status   Role
1    1          Ready    Active Master
2    2          Ready    Member
```

Verificar también configuración activa:

```text
awplus# show running-config
```

Debe observarse algo equivalente a:

```text
switch 1 provision x950-28
switch 2 provision x950-28

interface port1.0.33,port1.0.37
 stackport

interface port2.0.33,port2.0.37
 stackport
```


## 9. Alta disponibilidad y balanceo de carga

### Alta disponibilidad

Al activar VCStack, los dos switches operan como un único switch lógico con un **Master** y un **Member**. Si el Master falla, el otro miembro puede continuar operando como parte del stack.

Los dos cables de stack de 100G aportan:

```text
- Redundancia del enlace de stack
- Mayor ancho de banda interno entre switches
- Failover de administración entre Master y Member
- Posibilidad de configurar LAG distribuido entre ambos switches
```

### Balanceo de carga

El balanceo de carga **no queda automáticamente activo para todos los puertos de usuario** solamente por activar VCStack.

Para obtener balanceo de carga hacia servidores, firewalls, routers u otros switches, se debe configurar una **agregación de enlaces**, por ejemplo un **LAG / port-channel / trunk**, distribuyendo enlaces físicos entre ambos miembros del stack.

Ejemplo:

```text
Servidor / Firewall / Core
      |              |
      |              |
Switch 1          Switch 2
   \                /
    \____ VCStack__/
```


